# Anchors for Explainability in Machine Learning
### *Chapter 3 — XAI Techniques | Explainable AI in Medical Systems*

---

**Anchors** (Ribeiro, Singh & Guestrin, 2018) is a post-hoc, model-agnostic explanation method
that produces **high-precision IF-THEN rules** — called anchors — that are sufficient conditions
for a model's prediction. An anchor is defined as a rule $A$ such that:

$$\text{Precision}(A) = P(f(z) = f(x) \mid A(z) = 1) \geq \tau$$

where $f$ is the model, $x$ is the instance being explained, $z$ is a perturbed neighbour,
and $\tau$ is the precision threshold (default 0.95). The rule holds *regardless of what values
the non-anchored features take* — making it a stronger, more actionable statement than SHAP
or LIME, which only provide weighted attributions.

### Key properties

| Property | Description |
|---|---|
| **Precision** | Fraction of neighbourhood instances where the rule predicts the same class |
| **Coverage** | Fraction of all instances to which the rule applies |
| **Sufficiency** | The anchor is a sufficient condition for the prediction — not necessary |
| **Locality** | One anchor explains one prediction — different patients may get different rules |

### Anchors vs. LIME vs. SHAP

Unlike SHAP (which assigns a numerical attribution to each feature) and LIME (which fits a
local linear surrogate), Anchors produce a *rule* — a discrete, human-readable IF-THEN
statement. This makes them particularly suited to clinical contexts where clinicians need to
understand *which conditions are sufficient* to trigger a model's recommendation, and where
the explanation must be auditable by non-statisticians.

---

## Contents

1. [Setup and dataset](#1)
2. [AnchorTabularExplainer — setup](#2)
3. [Single patient explanation](#3)
4. [Comparing anchors across patients](#4)
5. [Precision, coverage and rule complexity](#5)
6. [RF vs GBM: do different models produce different anchors?](#6)
7. [AnchorText — clinical note classification](#7)
8. [Summary and clinical considerations](#8)


<a id='1'></a>
## 1. Setup and dataset


In [ ]:
# Install required packages (run once)
# !pip install anchor-exp scikit-learn pandas numpy matplotlib spacy
# !python -m spacy download en_core_web_sm


In [ ]:
from anchor import anchor_tabular, anchor_text
import spacy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, classification_report

warnings.filterwarnings('ignore')
print('All packages imported successfully.')


In [ ]:
# ── Wisconsin Breast Cancer Dataset ──────────────────────────────────────────
data = load_breast_cancer()
X    = pd.DataFrame(data.data, columns=data.feature_names)
y    = pd.Series(data.target, name='diagnosis')  # 0=malignant, 1=benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

feat_names  = list(X_train.columns)
class_names = ['malignant', 'benign']

print(f'Training set : {X_train.shape[0]} samples  |  Test set : {X_test.shape[0]} samples')
print(f'Features     : {X_train.shape[1]}')


In [ ]:
# ── Train models ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f'Random Forest AUC-ROC : {auc_rf:.4f}')

gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)
auc_gbm = roc_auc_score(y_test, gbm.predict_proba(X_test)[:, 1])
print(f'GBM AUC-ROC           : {auc_gbm:.4f}')


<a id='2'></a>
## 2. AnchorTabularExplainer — setup

The tabular explainer requires the **training data** to estimate the marginal distribution
of each feature for perturbation. The algorithm generates neighbours by sampling from these
distributions and uses a **beam search** with a multi-armed bandit to find the shortest
anchor rule that achieves the target precision threshold.

Key parameters in `explain_instance`:
- `threshold` (default 0.95): minimum required precision; higher = more conservative rules
- `max_anchor_size`: maximum number of conditions in the rule; `None` = no limit
- `beam_size`: width of the beam search; larger = better rules but slower


In [ ]:
# ── Initialise AnchorTabularExplainer ────────────────────────────────────────
explainer_tab = anchor_tabular.AnchorTabularExplainer(
    class_names       = class_names,
    feature_names     = feat_names,
    train_data        = X_train.values,
    categorical_names = {}   # empty dict = all features are continuous
)
print('AnchorTabularExplainer initialised.')
print(f'Class names   : {explainer_tab.class_names}')
print(f'Features      : {len(feat_names)}')


<a id='3'></a>
## 3. Single patient explanation

An anchor explanation for one patient answers:
*which feature conditions are sufficient for the model to predict this class,*
*regardless of how other features vary?*

The explanation is read as an IF-THEN rule:
IF `worst radius > 16.2` AND `worst concave points > 0.14` THEN predict: malignant
with precision 0.97 — meaning that among all patients satisfying these two conditions,
97% receive the same prediction from the model.


In [ ]:
# ── Explain a single test instance ───────────────────────────────────────────
patient_idx = 3
true_label  = class_names[y_test.iloc[patient_idx]]
pred_idx    = rf.predict(X_test.iloc[[patient_idx]])[0]
pred_label  = class_names[pred_idx]
pred_prob   = rf.predict_proba(X_test.iloc[[patient_idx]])[0]

exp = explainer_tab.explain_instance(
    X_test.values[patient_idx],
    rf.predict,
    threshold       = 0.95,
    max_anchor_size = None
)

print(f'Patient {patient_idx}')
print(f'  True label      : {true_label}')
print(f'  Predicted       : {pred_label}  (P(benign)={pred_prob[1]:.4f})')
print()
print('Anchor explanation:')
print(f'  IF {chr(10)+"  AND ".join(exp.names())}')
print(f'  THEN predict: {pred_label}')
print()
print(f'  Precision : {exp.precision():.4f}  (target >= 0.95)')
print(f'  Coverage  : {exp.coverage():.4f}  ({exp.coverage()*100:.1f}% of patients match this rule)')
print(f'  Rule count: {len(exp.names())} condition(s)')


In [ ]:
# ── Visualise the anchor explanation ─────────────────────────────────────────
rules = exp.names()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left panel: anchor rules displayed as horizontal bars
if rules:
    axes[0].barh(range(len(rules)), [1]*len(rules),
                 color='#2E75B6', alpha=0.75, height=0.45)
    axes[0].set_yticks(range(len(rules)))
    axes[0].set_yticklabels([f'IF  {r}' for r in rules],
                              fontsize=9.5, fontfamily='monospace')
    axes[0].set_xlim(0, 1.6)
    axes[0].axvline(1.0, color='#1F3864', linewidth=1, linestyle='--', alpha=0.4)
    axes[0].text(1.02, -0.6, f'THEN predict: {pred_label}',
                 fontsize=9, color='#1F3864', fontweight='bold')
axes[0].axis('off')
axes[0].set_title(
    f'Anchor Rules — Patient {patient_idx}\n'
    f'True: {true_label} | Predicted: {pred_label} | P(benign)={pred_prob[1]:.3f}',
    fontsize=9.5
)

# Right panel: precision and coverage metrics
metrics  = ['Precision', 'Coverage']
vals     = [exp.precision(), exp.coverage()]
bar_cols = ['#2E75B6', '#E8A020']
bars = axes[1].bar(metrics, vals, color=bar_cols, width=0.4)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Value', fontsize=10)
axes[1].set_title('Anchor Quality Metrics', fontsize=10)
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[1].axhline(0.95, color='gray', linestyle='--', linewidth=1, label='Threshold (0.95)')
axes[1].legend(fontsize=8.5)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


<a id='4'></a>
## 4. Comparing anchors across patients

Because Anchors are local explanations, different patients may receive completely different
rules — even when the model predicts the same class. This locality is a strength in clinical
contexts: it means the explanation reflects *this patient's specific situation* rather than
a population average. The comparison below shows how the rules vary across six patients.


In [ ]:
# ── Compare anchor rules for 6 patients ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i in range(4):
    e         = explainer_tab.explain_instance(
        X_test.values[i], rf.predict, threshold=0.95
    )
    pred_lbl  = class_names[rf.predict(X_test.iloc[[i]])[0]]
    true_lbl  = class_names[y_test.iloc[i]]
    prob      = rf.predict_proba(X_test.iloc[[i]])[0]
    rules_i   = e.names()
    ax        = axes[i]

    if rules_i:
        bar_col = '#2E75B6' if pred_lbl == 'benign' else '#C0392B'
        ax.barh(range(len(rules_i)), [1]*len(rules_i),
                color=bar_col, alpha=0.75, height=0.45)
        ax.set_yticks(range(len(rules_i)))
        ax.set_yticklabels(rules_i, fontsize=7.5)
    else:
        ax.text(0.5, 0.5, 'No anchor found', ha='center', va='center',
                transform=ax.transAxes, fontsize=10)
    ax.set_xlim(0, 1.5)
    ax.axis('off')
    ax.set_title(
        f'Patient {i} | True: {true_lbl} | Pred: {pred_lbl}\n'
        f'P(B)={prob[1]:.2f} | Prec={e.precision():.2f} | Cov={e.coverage():.3f}',
        fontsize=7.5
    )

b_p = mpatches.Patch(color='#2E75B6', label='Predicted: benign')
r_p = mpatches.Patch(color='#C0392B', label='Predicted: malignant')
fig.legend(handles=[b_p, r_p], loc='lower center', ncol=2,
           fontsize=9, bbox_to_anchor=(0.5, -0.03))
fig.suptitle(
    'Anchor Explanations — 6 Test Patients (Random Forest, Breast Cancer Dataset)\n'
    'Each patient receives a different rule reflecting their individual feature profile',
    fontsize=10, y=1.02
)
plt.tight_layout()
plt.show()


<a id='5'></a>
## 5. Precision, coverage and rule complexity

The precision-coverage trade-off is a fundamental property of Anchor explanations:
- **High precision, low coverage**: a very specific rule (many conditions) that applies
  to few patients but is almost always correct
- **Lower precision, high coverage**: a general rule (few conditions) that applies to
  many patients but with a smaller guarantee

The threshold parameter controls where on this curve the algorithm stops searching.
In clinical settings, high precision is usually preferable — a clinician needs to trust
that the rule holds for this patient, even if it applies to few others.


In [ ]:
# ── Precision, coverage and rule complexity across 15 patients ───────────────
precisions, coverages, n_rules_list, pred_labels_list = [], [], [], []

print('Computing anchors for 15 patients...')
for i in range(8):
    e = explainer_tab.explain_instance(
        X_test.values[i], rf.predict, threshold=0.95
    )
    precisions.append(e.precision())
    coverages.append(e.coverage())
    n_rules_list.append(len(e.names()))
    pred_labels_list.append(class_names[rf.predict(X_test.iloc[[i]])[0]])

print(f'Mean precision : {np.mean(precisions):.4f}')
print(f'Mean coverage  : {np.mean(coverages):.4f}')
print(f'Mean rule count: {np.mean(n_rules_list):.2f}')


In [ ]:
# ── Visualise precision, coverage and complexity ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

patient_ids = range(8)
pred_cols   = ['#2E75B6' if p == 'benign' else '#C0392B' for p in pred_labels_list]

# Precision
axes[0].bar(patient_ids, precisions, color=pred_cols)
axes[0].axhline(0.95, color='gray', linestyle='--', linewidth=1, label='Threshold (0.95)')
axes[0].set_ylim(0.85, 1.05)
axes[0].set_xlabel('Patient index', fontsize=9)
axes[0].set_ylabel('Precision', fontsize=9)
axes[0].set_title('Anchor Precision per Patient', fontsize=10)
axes[0].legend(fontsize=8)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Coverage
axes[1].bar(patient_ids, coverages, color='#E8A020')
axes[1].set_ylim(0, max(coverages) * 1.3 if max(coverages) > 0 else 0.5)
axes[1].set_xlabel('Patient index', fontsize=9)
axes[1].set_ylabel('Coverage', fontsize=9)
axes[1].set_title('Anchor Coverage per Patient', fontsize=10)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# Rule complexity
axes[2].bar(patient_ids, n_rules_list, color='#1EAAAA')
axes[2].set_xlabel('Patient index', fontsize=9)
axes[2].set_ylabel('Number of conditions', fontsize=9)
axes[2].set_title('Anchor Rule Complexity per Patient', fontsize=10)
axes[2].yaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

b_p = mpatches.Patch(color='#2E75B6', label='Predicted: benign')
r_p = mpatches.Patch(color='#C0392B', label='Predicted: malignant')
fig.legend(handles=[b_p, r_p], loc='lower center', ncol=2,
           fontsize=9, bbox_to_anchor=(0.5, -0.1))
fig.suptitle('Anchor Quality Metrics Across 15 Test Patients',
             fontsize=11, y=1.04)
plt.tight_layout()
plt.show()


<a id='6'></a>
## 6. RF vs GBM: do different models produce different anchors?

A key test of Anchor explanations is whether the same patient receives the same rule
from two different models that make the same prediction. Because Anchors interrogate the
model's decision boundary — not its internal weights — two models may identify different
sufficient conditions for the same outcome. Agreement between models suggests a robust,
data-driven rule; disagreement reveals model-specific decision boundaries.


In [ ]:
# ── Compare anchor rules: RF vs GBM for 4 patients ───────────────────────────
n_compare = 2
fig, axes = plt.subplots(n_compare, 2, figsize=(14, 3*n_compare))

for i in range(n_compare):
    for col, (model_name, model) in enumerate([('Random Forest', rf), ('GBM', gbm)]):
        e         = explainer_tab.explain_instance(
            X_test.values[i], model.predict, threshold=0.95
        )
        pred_lbl  = class_names[model.predict(X_test.iloc[[i]])[0]]
        true_lbl  = class_names[y_test.iloc[i]]
        rules_m   = e.names()
        ax        = axes[i, col]
        bar_col   = '#2E75B6' if pred_lbl == 'benign' else '#C0392B'

        if rules_m:
            ax.barh(range(len(rules_m)), [1]*len(rules_m),
                    color=bar_col, alpha=0.75, height=0.45)
            ax.set_yticks(range(len(rules_m)))
            ax.set_yticklabels(rules_m, fontsize=8)
        else:
            ax.text(0.5, 0.5, 'No anchor found', ha='center', va='center',
                    transform=ax.transAxes)
        ax.set_xlim(0, 1.5)
        ax.axis('off')
        ax.set_title(
            f'{model_name} | Patient {i} | True: {true_lbl} | Pred: {pred_lbl}\n'
            f'Precision={e.precision():.3f}  Coverage={e.coverage():.3f}  Rules={len(rules_m)}',
            fontsize=8.5
        )

fig.suptitle('Anchor Rules: Random Forest vs GBM for the Same 4 Patients',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()


<a id='7'></a>
## 7. AnchorText — clinical note classification

`AnchorText` applies the Anchor framework to text classifiers by perturbing documents
at the **word level** — replacing words with unknown tokens or synonyms — and identifying
which words, when present, are sufficient to determine the model's prediction.

The result is a rule such as:
*IF 'biopsy' present AND 'malignant' present THEN predict: Oncology (precision = 1.0)*

This is clinically meaningful: it reveals the specific terms that the model uses as decision
anchors, which can be audited against clinical knowledge to verify that the model is using
appropriate medical terminology rather than spurious correlations.


In [ ]:
# ── Build synthetic clinical note dataset ─────────────────────────────────────
cardiac_notes = [
    'patient presents with chest pain radiating to left arm shortness of breath',
    'severe chest tightness diaphoresis nausea typical cardiac event',
    'myocardial infarction suspected chest pain elevated troponin',
    'palpitations chest discomfort irregular heartbeat ecg st elevation',
    'heart failure dyspnoea orthopnoea peripheral oedema',
    'angina pectoris exertional chest pain relieved rest nitrates',
    'atrial fibrillation rapid ventricular rate chest discomfort',
    'cardiomyopathy reduced ejection fraction exertional dyspnoea',
    'hypertensive crisis headache elevated blood pressure chest pain',
    'pericarditis pleuritic chest pain pericardial friction rub',
    'ventricular tachycardia amiodarone defibrillation cardiac arrest',
    'coronary artery disease stent placement percutaneous intervention',
]

oncology_notes = [
    'biopsy confirmed malignant tumour lymph node involvement stage three',
    'breast cancer diagnosis mammography core needle biopsy',
    'lung adenocarcinoma pleural effusion metastatic spread confirmed',
    'colorectal cancer elevated cea levels colonoscopy findings',
    'chemotherapy non-hodgkin lymphoma remission oncology',
    'prostate cancer elevated psa biopsy gleason score seven',
    'melanoma breslow thickness lymph node biopsy oncology',
    'ovarian cancer ca125 elevation peritoneal carcinomatosis',
    'glioblastoma multiforme resected radiotherapy temozolomide',
    'pancreatic ductal adenocarcinoma obstructive jaundice weight loss',
    'hepatocellular carcinoma alpha-fetoprotein liver cirrhosis biopsy',
    'cervical cancer staging mri positron emission tomography scan',
]

extras = ['', ' and fatigue', ' on examination', ' noted by clinician',
          ' requiring urgent assessment', ' with family history',
          ' after investigation', ' per clinical assessment']

def augment(templates, n=80):
    return [templates[i % len(templates)] + extras[i % len(extras)] for i in range(n)]

texts  = augment(cardiac_notes, 80) + augment(oncology_notes, 80)
labels = [0]*80 + [1]*80  # 0=Cardiology, 1=Oncology

X_txt_tr, X_txt_te, y_txt_tr, y_txt_te = train_test_split(
    texts, labels, test_size=0.20, random_state=42, stratify=labels
)
print(f'Training notes: {len(X_txt_tr)}  |  Test notes: {len(X_txt_te)}')


In [ ]:
# ── Train TF-IDF + Logistic Regression pipeline ───────────────────────────────
text_pipeline = make_pipeline(
    TfidfVectorizer(max_features=300, ngram_range=(1, 2), stop_words='english'),
    LogisticRegression(max_iter=500, random_state=42)
)
text_pipeline.fit(X_txt_tr, y_txt_tr)
print(f'Text classifier accuracy: {text_pipeline.score(X_txt_te, y_txt_te):.4f}')


In [ ]:
# ── Initialise AnchorText ─────────────────────────────────────────────────────
nlp = spacy.load('en_core_web_sm')

explainer_text = anchor_text.AnchorText(
    nlp                  = nlp,
    class_names          = ['Cardiology', 'Oncology'],
    use_unk_distribution = True   # replace non-anchor words with UNK tokens
)
print('AnchorText explainer initialised.')


In [ ]:
# ── Anchor explanations for 4 clinical notes (2 per class) ───────────────────
# Select 2 Cardiology and 2 Oncology notes from the test set
selected = []
for cls_idx, cls_name in [(0, 'Cardiology'), (1, 'Oncology')]:
    idxs = [i for i, l in enumerate(y_txt_te) if l == cls_idx][:2]
    for idx in idxs:
        selected.append((X_txt_te[idx], y_txt_te[idx], cls_name, idx))

print('Computing text anchors for 4 clinical notes...')
text_exps = []
for note, true_lbl_idx, true_name, idx in selected:
    exp_t = explainer_text.explain_instance(
        note,
        text_pipeline.predict,
        threshold = 0.95,
        use_proba = False
    )
    pred_cls  = text_pipeline.predict([note])[0]
    pred_name = ['Cardiology', 'Oncology'][pred_cls]
    text_exps.append((note, true_name, pred_name, exp_t))
    print(f'\nNote: {note[:60]}...')
    print(f'  True: {true_name} | Predicted: {pred_name}')
    anchor_rule = ' AND '.join(exp_t.names()) if exp_t.names() else '(no anchor found)'
    print(f'  Anchor: IF {anchor_rule} THEN {pred_name}')
    print(f'  Precision: {exp_t.precision():.4f} | Coverage: {exp_t.coverage():.4f}')


In [ ]:
# ── Visualise text anchor explanations ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax_i, (note, true_name, pred_name, exp_t) in enumerate(text_exps):
    words = exp_t.names()
    prec  = exp_t.precision()
    cov   = exp_t.coverage()
    ax    = axes[ax_i]
    col   = '#2E75B6' if pred_name == 'Cardiology' else '#C0392B'

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # Prediction header
    ax.text(0.04, 0.92, f'Predicted: {pred_name}',
            fontsize=10.5, fontweight='bold', color=col, transform=ax.transAxes)

    # Anchor rule
    ax.text(0.04, 0.78, 'Anchor rule:',
            fontsize=9, color='gray', transform=ax.transAxes)
    if words:
        rule_text = '  IF  ' + chr(10) + '  AND '.join(words)
    else:
        rule_text = '  (no anchor found)'
    ax.text(0.04, 0.62, rule_text, fontsize=9, fontfamily='monospace',
            color='#1F3864', transform=ax.transAxes, linespacing=1.8)

    # Metrics
    ax.text(0.04, 0.32,
            f'Precision = {prec:.3f}  |  Coverage = {cov:.3f}',
            fontsize=8.5, color='gray', transform=ax.transAxes)

    # Note preview
    ax.text(0.04, 0.15, f'Note: {note[:72]}...',
            fontsize=7.5, color='#666', transform=ax.transAxes,
            style='italic', wrap=True)

    ax.set_title(f'True class: {true_name}', fontsize=9, color='#555')
    # Border
    for spine_name in ['top','bottom','left','right']:
        ax.spines[spine_name].set_visible(True)
        ax.spines[spine_name].set_color('#DDDDDD')

fig.suptitle(
    'Anchor Text Explanations — Clinical Note Classification\n'
    'Words in the anchor are sufficient for the prediction to hold',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()


<a id='8'></a>
## 8. Summary and clinical considerations

### Key properties demonstrated

1. **Anchors produce IF-THEN rules, not numerical weights.** This is the fundamental
   distinction from SHAP and LIME. A clinician can read `IF worst radius > 16.2 AND
   worst concave points > 0.14 THEN malignant` and immediately understand what conditions
   are driving the prediction — without needing to interpret a bar chart of attribution values.

2. **Precision is guaranteed at the specified threshold.** An anchor with precision 0.97
   means that 97% of patients satisfying the rule conditions receive the same prediction.
   This makes the rule auditable and regulatorily accountable in a way that a weighted
   attribution is not.

3. **Coverage quantifies generalisability.** A rule with very low coverage applies to few
   patients — it may be highly precise but clinically niche. High-coverage rules are more
   general but typically require fewer conditions.

4. **Locality means different patients get different rules.** This is a strength, not a
   weakness: in clinical medicine, the conditions sufficient to predict malignancy for a
   young patient may differ from those for an elderly one.

5. **Text anchors reveal which words the model depends on.** In clinical NLP applications,
   this enables a direct audit of whether the model is using appropriate clinical terminology
   or has learned spurious correlations from training data.

### Clinical limitations

- **Anchors are sufficient conditions, not necessary ones.** The rule `IF worst radius > 16.2`
  being an anchor does not mean that *only* patients with worst radius > 16.2 will be
  predicted malignant — other conditions may also be sufficient.
- **Coverage can be very low** for complex cases near the decision boundary, producing rules
  that apply to very few patients and are hard to generalise.
- **Computation time** scales with the complexity of the rule search; for large feature spaces
  or low-threshold requirements, explanations may take minutes per instance.
- **Stability** is not guaranteed across runs due to the stochastic sampling in the beam search.

---

## Further reading

- Ribeiro, M.T., Singh, S. & Guestrin, C. (2018). *Anchors: High-precision model-agnostic
  explanations.* AAAI 2018, pp. 1527-1535.
- Anchor GitHub: https://github.com/marcotcr/anchor
